# 04 — Evaluation & Comparison
**Drone Detection | ITMO Diploma Thesis**

**Модели в сводке:** все ключи `yolo12n` / `yolo12s` / `yolo26n` / `yolo26s` из `yolo_metrics.json`, плюс **Faster R-CNN**, если рядом лежит `rcnn_metrics.json` (скопируйте из обучения `03_faster_rcnn_train.ipynb`).

**Пути (CELL 3):**
- Локально по умолчанию: `WEIGHTS_DIR` = `POSTwork/weights/weights`, `RESULTS_DIR` = `POSTwork/results/results`, датасет YOLO для `val()` = `PREwork/dataset/prepared/yolo`.
- Google Colab по умолчанию: прежняя схема `MyDrive/Colab Notebooks/{prepared,weights,results}`.
- Переключение: переменная окружения `DRONE_EVAL_LAYOUT=postwork` или `drive`.

Per-class mAP для YOLO: при наличии GPU выполняется `model.val(test)`; иначе берутся списки `per_class_mAP50` из JSON.

Метрики: mAP@0.5, mAP@0.5:0.95, Precision, Recall, FPS, Size

In [ ]:
# ── CELL 1: Install & GPU ─────────────────────────────────────────────────────
!uv pip install ultralytics pycocotools seaborn scikit-learn
import ultralytics; ultralytics.checks()
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU!'
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── CELL 2: Mount Drive (только Google Colab) ─────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Локальный запуск: монтирование Drive не требуется (пути в CELL 3).')

In [ ]:
# ── CELL 3: Imports & paths ───────────────────────────────────────────────────
import json
import os
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from ultralytics import YOLO
from tqdm import tqdm


def _find_repo_root() -> Path:
    """Корень репозитория (рядом PREwork/ и POSTwork/)."""
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / 'PREwork').is_dir() and (p / 'POSTwork').is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd().resolve()


REPO_ROOT = _find_repo_root()
try:
    import google.colab  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

# postwork: веса/метрики из POSTwork; drive: классическая схема Colab Notebooks
_USE_POSTWORK = os.environ.get('DRONE_EVAL_LAYOUT', 'postwork' if not _IN_COLAB else 'drive') == 'postwork'

if _USE_POSTWORK:
    DATA_DIR = REPO_ROOT / 'PREwork' / 'dataset' / 'prepared' / 'yolo'
    COCO_DIR = REPO_ROOT / 'PREwork' / 'dataset' / 'prepared' / 'dataset_coco'
    WEIGHTS_DIR = REPO_ROOT / 'POSTwork' / 'weights' / 'weights'
    RESULTS_DIR = REPO_ROOT / 'POSTwork' / 'results' / 'results'
else:
    DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks')
    DATA_DIR = DRIVE_ROOT / 'prepared' / 'yolo'
    COCO_DIR = DRIVE_ROOT / 'prepared' / 'dataset_coco'
    WEIGHTS_DIR = DRIVE_ROOT / 'weights'
    RESULTS_DIR = DRIVE_ROOT / 'results'

YAML_PATH = DATA_DIR / 'data.yaml'

CLASS_NAMES = ['DRONE', 'AIRPLANE', 'HELICOPTER', 'BIRD']
CLASS_COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f1c40f']
MODEL_COLORS = {
    'YOLOv12n': '#e74c3c',
    'YOLOv12s': '#3498db',
    'YOLO26n': '#8e44ad',
    'YOLO26s': '#16a085',
    'Faster R-CNN': '#27ae60',
}
YOLO_WEIGHT_FILES = [
    ('YOLOv12n', 'yolo12n_drone_best.pt'),
    ('YOLOv12s', 'yolo12s_drone_best.pt'),
    ('YOLO26n', 'yolo26n_drone_best.pt'),
    ('YOLO26s', 'yolo26s_drone_best.pt'),
]

if YAML_PATH.exists():
    _r = DATA_DIR.resolve()
    _fix = (
        f'path: {_r.as_posix()}\n'
        'train: images/train\nval: images/val\ntest: images/test\n'
        'nc: 4\nnames: [\'DRONE\', \'AIRPLANE\', \'HELICOPTER\', \'BIRD\']\n'
    )
    YAML_PATH.write_text('# Patched for eval\n' + _fix, encoding='utf-8')

print(f'REPO_ROOT={REPO_ROOT}')
print(f'Layout: {"POSTwork + PREwork/dataset" if _USE_POSTWORK else "Drive Colab Notebooks"}')
print(f'WEIGHTS_DIR={WEIGHTS_DIR}')
print(f'RESULTS_DIR={RESULTS_DIR}')
print(f'data.yaml exists: {YAML_PATH.exists()}')

## 1. Загрузка метрик

In [ ]:
# ── CELL 4: Load metrics + per-class mAP (val на GPU или из JSON) ───────────────
_yolo_path = RESULTS_DIR / 'yolo_metrics.json'
if not _yolo_path.exists():
    raise FileNotFoundError(f'Нет {_yolo_path}. Положите yolo_metrics.json в RESULTS_DIR.')

with open(_yolo_path, encoding='utf-8') as f:
    yolo_data = json.load(f)

_rcnn_path = RESULTS_DIR / 'rcnn_metrics.json'
if _rcnn_path.exists():
    with open(_rcnn_path, encoding='utf-8') as f:
        rcnn_data = json.load(f)
else:
    rcnn_data = {}
    print(f'Нет {_rcnn_path} — Faster R-CNN в таблицах/графиках будет пропущен (скопируйте с обучения).')

_JSON_KEY = {
    'YOLOv12n': 'yolo12n',
    'YOLOv12s': 'yolo12s',
    'YOLO26n': 'yolo26n',
    'YOLO26s': 'yolo26s',
}

yolo_val_results: dict = {}
per_class_by_yolo: dict = {}


def _per_class_from_json(jk: str) -> dict | None:
    d = yolo_data.get(jk)
    if not d:
        return None
    pc = d.get('per_class_mAP50')
    if isinstance(pc, list) and len(pc) == len(CLASS_NAMES):
        return dict(zip(CLASS_NAMES, pc))
    return None


for label, fname in YOLO_WEIGHT_FILES:
    jk = _JSON_KEY[label]
    if jk not in yolo_data:
        print(f'Skip {label}: нет ключа {jk!r} в yolo_metrics.json')
        continue
    wp = WEIGHTS_DIR / fname
    ran_val = False
    if wp.exists() and YAML_PATH.exists() and torch.cuda.is_available():
        try:
            model = YOLO(str(wp))
            val = model.val(data=str(YAML_PATH), split='test', device=0, verbose=False)
            yolo_val_results[label] = val
            per_class_by_yolo[label] = dict(zip(CLASS_NAMES, val.box.ap50.tolist()))
            ran_val = True
            print(f'Per-class mAP@0.5 (val) — {label}:', per_class_by_yolo[label])
        except Exception as exc:
            print(f'val() не удался для {label}: {exc}. Берём per-class из JSON.')
    if not ran_val:
        fallback = _per_class_from_json(jk)
        if fallback is not None:
            per_class_by_yolo[label] = fallback
            print(f'Per-class mAP@0.5 (JSON) — {label}:', fallback)
        else:
            print(f'Нет per-class для {label} (ни val, ни per_class_mAP50 в JSON).')

## 2. Сводная таблица метрик

In [ ]:
# ── CELL 5: Summary table — все модели из yolo_metrics.json + опционально R-CNN ─
_YOLO_ORDER = [
    ('YOLOv12n', 'yolo12n'),
    ('YOLOv12s', 'yolo12s'),
    ('YOLO26n', 'yolo26n'),
    ('YOLO26s', 'yolo26s'),
]

rows = []
for label, key in _YOLO_ORDER:
    if key not in yolo_data:
        continue
    d = yolo_data[key]
    if not isinstance(d, dict) or 'mAP50' not in d:
        continue
    rows.append({
        'Model': label,
        'mAP@0.5':      d['mAP50'],
        'mAP@0.5:0.95': d['mAP50_95'],
        'Precision':    d.get('precision', 'N/A'),
        'Recall':       d.get('recall', 'N/A'),
        'FPS (T4)':     d['fps'],
        'Size (MB)':    d['size_mb'],
    })

if rcnn_data and 'mAP50' in rcnn_data:
    _rcnn_prec = rcnn_data.get('precision')
    _rcnn_rec = rcnn_data.get('recall')
    rows.append({
        'Model': 'Faster R-CNN',
        'mAP@0.5':      rcnn_data['mAP50'],
        'mAP@0.5:0.95': rcnn_data['mAP50_95'],
        'Precision':    _rcnn_prec if _rcnn_prec is not None else 'N/A',
        'Recall':       _rcnn_rec if _rcnn_rec is not None else 'N/A',
        'FPS (T4)':     rcnn_data['fps'],
        'Size (MB)':    rcnn_data['size_mb'],
    })
else:
    print('Faster R-CNN: нет rcnn_metrics.json или поля mAP50 — строка не добавлена.')

if not rows:
    raise ValueError('Нет ни одной строки для сводной таблицы. Проверьте yolo_metrics.json / rcnn_metrics.json.')

df_summary = pd.DataFrame(rows).set_index('Model')

print('\n' + '='*65)
print('MODEL COMPARISON SUMMARY')
print('='*65)
print(df_summary.to_string())

df_summary.to_csv(RESULTS_DIR / 'model_comparison.csv')
print(f'\nSaved → {RESULTS_DIR / "model_comparison.csv"}')

In [ ]:
# ── CELL 6: Bar chart — mAP comparison ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(max(13, 2 + len(df_summary) * 1.2), 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

models = list(df_summary.index)
colors = [MODEL_COLORS.get(m, '#7f8c8d') for m in models]

map50_vals    = [float(df_summary.loc[m, 'mAP@0.5'])       for m in models]
map5095_vals  = [float(df_summary.loc[m, 'mAP@0.5:0.95'])    for m in models]
fps_vals      = [float(df_summary.loc[m, 'FPS (T4)'])       for m in models]

x = np.arange(len(models))
width = 0.35

b1 = axes[0].bar(x - width/2, map50_vals,   width, label='mAP@0.5',    color=colors, alpha=0.85)
b2 = axes[0].bar(x + width/2, map5095_vals, width, label='mAP@0.5:0.95', color=colors, alpha=0.5)
axes[0].set_title('Accuracy (mAP)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=15, ha='right')
axes[0].set_ylabel('mAP')
axes[0].set_ylim(0, 1.0)
axes[0].legend(['mAP@0.5', 'mAP@0.5:0.95'])
axes[0].grid(True, axis='y', alpha=0.3)
for bar, val in zip(b1, map50_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=8)

axes[1].bar(models, fps_vals, color=colors, alpha=0.85, edgecolor='white')
axes[1].set_title('Inference Speed (FPS on T4)')
axes[1].set_ylabel('FPS')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(True, axis='y', alpha=0.3)
for i, (m, v) in enumerate(zip(models, fps_vals)):
    axes[1].text(i, v + 1, str(v), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'model_comparison.png', dpi=150)
plt.show()

In [ ]:
# ── CELL 7: Per-class mAP@0.5 (YOLO: val/JSON + R-CNN из rcnn_metrics.json) ───
per_class_plot = dict(per_class_by_yolo)
_ap = rcnn_data.get('per_class_mAP50') if rcnn_data else None
if isinstance(_ap, list) and len(_ap) == len(CLASS_NAMES):
    per_class_plot['Faster R-CNN'] = dict(zip(CLASS_NAMES, _ap))

if not per_class_plot:
    print('Skip per-class plot: нет YOLO-весов и нет per_class_mAP50 в rcnn_metrics.json.')
else:
    fig, ax = plt.subplots(figsize=(max(11, 2 + len(per_class_plot) * 2), 5))

    n_classes = len(CLASS_NAMES)
    x = np.arange(n_classes)
    labels_order = list(per_class_plot.keys())
    n_m = len(labels_order)
    width = min(0.8 / max(n_m, 1), 0.22)
    offsets = [width * (i - (n_m - 1) / 2) for i in range(n_m)]

    for off, lab in zip(offsets, labels_order):
        ap = [per_class_plot[lab].get(c, 0) for c in CLASS_NAMES]
        ax.bar(x + off, ap, width, label=lab, color=MODEL_COLORS.get(lab, '#95a5a6'), alpha=0.88)

    ax.set_title('Per-Class mAP@0.5', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, fontsize=11)
    ax.set_ylabel('AP@0.5')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'per_class_map.png', dpi=150)
    plt.show()

## 3. Confusion Matrix

In [ ]:
# ── CELL 8: Confusion matrix ───────────────────────────────────────────────────
def get_yolo_confusion(weights_path: Path, data_yaml: Path,
                       conf: float = 0.25, iou: float = 0.5) -> np.ndarray:
    """Compute confusion matrix for YOLO model.

    Args:
        weights_path: Path to model weights.
        data_yaml: Path to data.yaml.
        conf: Confidence threshold.
        iou: IoU threshold for matching.

    Returns:
        Normalized confusion matrix (n_classes x n_classes).
    """
    from ultralytics.utils.metrics import ConfusionMatrix
    model = YOLO(str(weights_path))
    results = model.val(data=str(data_yaml), split='test',
                        device=0, conf=conf, iou=iou, verbose=False)
    cm = results.confusion_matrix.matrix
    # Normalize per row
    row_sums = cm.sum(axis=1, keepdims=True).clip(min=1)
    return cm / row_sums


_yolo_for_cm = [(lab, WEIGHTS_DIR / fn) for lab, fn in YOLO_WEIGHT_FILES
                if (WEIGHTS_DIR / fn).exists()]
_run_cm = bool(_yolo_for_cm) and YAML_PATH.exists() and torch.cuda.is_available()
if not _yolo_for_cm:
    print('Пропуск confusion matrices: нет .pt в WEIGHTS_DIR.')
elif not YAML_PATH.exists():
    print('Пропуск confusion matrices: нет data.yaml (PREwork/dataset/prepared/yolo).')
elif not torch.cuda.is_available():
    print('Пропуск confusion matrices: нужен CUDA (model.val на GPU).')
if _run_cm:
    n_cm = len(_yolo_for_cm)
    ncols = 2 if n_cm > 2 else n_cm
    nrows = int(np.ceil(n_cm / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()
    fig.suptitle('Confusion Matrices (normalized, test set)', fontsize=13, fontweight='bold')

    tick_labels = CLASS_NAMES + ['Background']
    for ax, (title, wpath) in zip(axes, _yolo_for_cm):
        cm = get_yolo_confusion(wpath, YAML_PATH)
        n = min(cm.shape[0], len(tick_labels))
        sns.heatmap(cm[:n, :n], annot=True, fmt='.2f', cmap='Blues', ax=ax,
                    xticklabels=tick_labels[:n], yticklabels=tick_labels[:n],
                    vmin=0, vmax=1, linewidths=0.5)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')

    for j in range(len(_yolo_for_cm), len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'confusion_matrices.png', dpi=150)
    plt.show()

## 4. Precision-Recall Curves

In [ ]:
# ── CELL 9: PR curves ─────────────────────────────────────────────────────────
def get_pr_data(weights_path: Path, data_yaml: Path) -> dict:
    """Extract precision-recall curve data from YOLO validation.

    Args:
        weights_path: Path to model weights.
        data_yaml: Path to data.yaml.

    Returns:
        Dict with precision/recall arrays per class.
    """
    model = YOLO(str(weights_path))
    results = model.val(data=str(data_yaml), split='test', device=0, verbose=False)
    return {
        'precision': results.box.p,
        'recall':    results.box.r,
        'ap50':      results.box.ap50,
    }


_yolo_pr = [(lab, WEIGHTS_DIR / fn) for lab, fn in YOLO_WEIGHT_FILES
            if (WEIGHTS_DIR / fn).exists()]
_run_pr = bool(_yolo_pr) and YAML_PATH.exists() and torch.cuda.is_available()
if not _yolo_pr:
    print('Пропуск PR-кривых: нет весов в WEIGHTS_DIR.')
elif not YAML_PATH.exists():
    print('Пропуск PR-кривых: нет data.yaml.')
elif not torch.cuda.is_available():
    print('Пропуск PR-кривых: нужен CUDA.')
if _run_pr:
    n_pr = len(_yolo_pr)
    ncols = 2 if n_pr > 2 else max(n_pr, 1)
    nrows = int(np.ceil(n_pr / ncols)) if n_pr else 1
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()
    fig.suptitle('Precision-Recall Curves (test set)', fontsize=13, fontweight='bold')

    for ax, (model_name, wpath) in zip(axes, _yolo_pr):
        pr_data = get_pr_data(wpath, YAML_PATH)
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            if cls_idx < len(pr_data['recall']):
                recall_arr = np.array(pr_data['recall'][cls_idx]) if hasattr(pr_data['recall'][cls_idx], '__len__') else [pr_data['recall'][cls_idx]]
                precision_arr = np.array(pr_data['precision'][cls_idx]) if hasattr(pr_data['precision'][cls_idx], '__len__') else [pr_data['precision'][cls_idx]]
                ap = pr_data['ap50'][cls_idx] if cls_idx < len(pr_data['ap50']) else 0
                ax.plot(recall_arr, precision_arr,
                        color=CLASS_COLORS[cls_idx], linewidth=2,
                        label=f'{cls_name} (AP={ap:.3f})')
        ax.set_title(model_name)
        ax.set_xlabel('Recall')
        ax.set_ylabel('Precision')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    for j in range(len(_yolo_pr), len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'pr_curves.png', dpi=150)
    plt.show()

In [ ]:
# ── CELL 10: Speed-Accuracy tradeoff scatter ──────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

model_data = []
for label, key in _YOLO_ORDER:
    if key not in yolo_data:
        continue
    d = yolo_data[key]
    if isinstance(d, dict) and 'mAP50' in d:
        model_data.append((label, d['fps'], d['mAP50'], d['size_mb']))
if rcnn_data and 'mAP50' in rcnn_data:
    model_data.append(('Faster R-CNN', rcnn_data['fps'], rcnn_data['mAP50'], rcnn_data['size_mb']))

if not model_data:
    print('Нет данных для scatter (проверьте yolo_metrics.json / rcnn_metrics.json).')
else:
    for name, fps, map50, size in model_data:
        color = MODEL_COLORS.get(name, '#34495e')
        ax.scatter(fps, map50, s=float(size) * 1.5, color=color, alpha=0.85,
                   edgecolors='black', linewidth=1.5, zorder=5)
        ax.annotate(name, (fps, map50), textcoords='offset points',
                    xytext=(8, 4), fontsize=9, color=color, fontweight='bold')
    ax.set_xlabel('FPS (T4 GPU)', fontsize=12)
    ax.set_ylabel('mAP@0.5', fontsize=12)
    ax.set_title('Speed vs. Accuracy Tradeoff\n(bubble size = model size MB)',
                 fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'speed_accuracy_tradeoff.png', dpi=150)
    plt.show()

In [ ]:
# ── CELL 11: Final summary print ──────────────────────────────────────────────
print('=' * 65)
print(f'EVALUATION COMPLETE — артефакты: {RESULTS_DIR}')
print('=' * 65)
saved = [
    'model_comparison.csv', 'model_comparison.png',
    'per_class_map.png', 'confusion_matrices.png',
    'pr_curves.png', 'speed_accuracy_tradeoff.png',
]
for f in saved:
    p = RESULTS_DIR / f
    exists = 'OK' if p.exists() else '--'
    print(f'  [{exists}] {f}')
print('\nNext: 05_video_inference.ipynb')